In [1]:
import pandas as pd
import os
from utils.dataframe_tools import global_stratified_split
from utils.dataframe_tools import truncate_to_block_by_schema, augment_main_block, create_specialist_dataset, create_specialist_dataset_intersect
from utils.pruning_and_merge import merge_field_blocks_tree_similarity

In [2]:
# ori_csv_path = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'completeness', 'dataset_29_completed_label.csv')
# output_dir_train_test = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test')
# output_dir_blocks_first = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test', 'blocks', 'first_truncation')
# output_dir_blocks_second = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test', 'blocks', 'merge')
ori_csv_path = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv', 'dataset_20_d2.csv')
output_dir_train_test = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv_merged', 'train_test')
output_dir_blocks_first = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv_merged', 'train_test', 'blocks', 'first_truncation')
output_dir_blocks_second = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv_merged', 'train_test', 'blocks', 'merge')

In [3]:
global_stratified_split(ori_csv_path, output_dir_train_test)

开始执行全局数据集分割...
输出目录已准备好: ..\TrafficData\dataset_20_d2_csv_merged\train_test
正在从 ..\TrafficData\dataset_20_d2_csv\dataset_20_d2.csv 加载总数据...
数据加载完成，共 6256032 条记录。

[步骤 1/2] 分割出 10% 的测试集...
[步骤 2/2] 从剩余数据中分割出 11.1% 的验证集 (相当于总数据的 10%)...

分割完成！各数据集规模如下:
 - 训练集 (Train Set): 5004824 条 (~80%)
 - 验证集 (Validation Set): 625604 条 (~10%)
 - 测试集 (Test Set): 625604 条 (~10%)

正在保存文件...
 - 训练集已保存到: ..\TrafficData\dataset_20_d2_csv_merged\train_test\train_set.csv
 - 验证集已保存到: ..\TrafficData\dataset_20_d2_csv_merged\train_test\validation_set.csv
 - 测试集已保存到: ..\TrafficData\dataset_20_d2_csv_merged\train_test\test_set.csv

全局数据集分割步骤已成功完成！


In [4]:
train_set_path = os.path.join(output_dir_train_test, 'train_set.csv')

truncate_to_block_by_schema(train_set_path, output_dir_blocks_first)

Data loading completed, 5004824 records here. 
Fingerprints has been built. 
There are 230 Field Block in total. 


Saving blocks: 100%|██████████| 230/230 [01:22<00:00,  2.80it/s]


Truncation succeeded! 


In [5]:
merge_field_blocks_tree_similarity(output_dir_blocks_first, output_dir_blocks_second, similarity_threshold=0.8)

开始执行基于【结构化树相似度】的Field Block合并与剪枝流程...

[步骤 1/5] 正在扫描并分析所有原始Block...


Scanning blocks: 100%|██████████| 230/230 [00:05<00:00, 45.64it/s]



[步骤 2/5] 识别完成:
 - 44 个核心专家 (样本数 >= 1000)
 - 186 个待合并的小型专家

[步骤 3/5] 正在为小型专家寻找最佳合并目标...


Merging small blocks: 100%|██████████| 186/186 [00:05<00:00, 31.06it/s]



[步骤 4/5] 正在处理 6 个“孤儿”Block...

[步骤 5/5] 正在保存 45 个最终的Block...


Saving final blocks: 100%|██████████| 45/45 [00:33<00:00,  1.33it/s]



合并与剪枝成功！最终生成了 45 个专家Block。
合并日志已保存到: ..\TrafficData\dataset_20_d2_csv_merged\train_test\blocks\merge\merge_log.json


In [7]:
# main_block_name = '24'
main_block_name = '34'
# output_path = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test', 'blocks', 'chief_block_v2.csv')
output_path = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv_merged', 'train_test', 'blocks', 'chief_block_dataset_20_d2.csv')
augment_main_block(output_dir_blocks_second, main_block_name, output_path)

开始执行“最大化覆盖”的数据补充策略...

[步骤 1/4] 加载 Main Block '34'...

[步骤 2/4] 扫描所有Block，建立类别分布情报库...


Scanning Blocks: 100%|██████████| 45/45 [00:07<00:00,  6.22it/s]



[步骤 3/4] 在 Main Block 中找到 3 个需要补充的类别:
['baidu', 'iqiyi', 'youku']

[步骤 4/4] 开始从所有其他Block中寻找并合并补充数据...


Augmenting Classes: 100%|██████████| 3/3 [00:23<00:00,  7.70s/it]



正在合并所有数据...

数据补充成功！
 - 原始 Main Block 样本数: 2623630
 - 补充后总样本数: 3311633
 - 最终数据集已保存到: ..\TrafficData\dataset_20_d2_csv_merged\train_test\blocks\chief_block_dataset_20_d2.csv


In [3]:
main_block_name = '24'
output_path = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test', 'blocks', 'specific_classes_intersec.csv')
create_specialist_dataset_intersect(output_dir_blocks_second, main_block_name, output_path, ['google', 'twitter'])

### 创建“共同独有特征专家模型”专用训练集 ###
### 目标类别: ['google', 'twitter'] ###

[步骤 1/5] 正在从所有Block中收集 google, twitter 的数据...


 -> 数据收集完成，共找到 20240 个相关样本。
 -> 数据来源于以下 6 个'捐献者'Block: {'44', '40', '39', '77', '35', '34'}

[步骤 2/5] 正在计算“共同的独有特征”...
 -> 计算完成，找到 6 个共同独有特征: ['tcp.option_kind', 'tcp.option_len', 'tcp.options.nop', 'tcp.options.timestamp', 'tcp.options.timestamp.tsecr', 'tcp.options.timestamp.tsval']

[步骤 3/5] 正在创建只包含独有特征的数据集...

[步骤 4/5] 正在保存最终的专家训练集...

专家训练集创建成功！
 - 总样本数: 20240
 - 总特征数: 6
 - 最终数据集已保存到: ..\TrafficData\dataset_29_d1_csv_merged\train_test\blocks\specific_classes_intersec.csv
